In [ ]:
import os
import gc

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, hsv_to_rgb
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import measure

import cfospy

In [ ]:
def mask_range(IDs, r):
    """Return padded bounding box (xmin, xmax, ymin, ymax, zmin, zmax) for given region IDs."""
    ID_li = []
    for rID in IDs:
        if not ca.smallID_q(rID):
            # For non-small IDs, get child and middle IDs and store as two list elements
            child_IDs, child_regions, middle_IDs, middle_regions = ca.get_child_IDs2(rID)
            ID_li += child_IDs, middle_IDs  # extend with two items (lists)
        else:
            # For small IDs, store the ID itself
            ID_li += [rID]

    # Use only the first element of ID_li (list of IDs or a single ID)
    mask0 = np.isin(
        np.swapaxes((ca.voxel_ID_order_all).reshape(ca.x_num, ca.y_num, ca.z_num), 0, 2),
        ID_li[0]
    )

    v_ind = np.where(mask0)

    xmin = np.min(v_ind[2])
    ymin = np.min(v_ind[1])
    zmin = np.min(v_ind[0])
    xmax = np.max(v_ind[2])
    ymax = np.max(v_ind[1])
    zmax = np.max(v_ind[0])

    # Pad the bounds by a fraction determined by r
    xmin = int(xmin - (xmax - xmin) / r)
    ymin = int(ymin - (ymax - ymin) / r)
    zmin = int(zmin - (zmax - zmin) / r)
    xmax = int(xmax + (xmax - xmin) / r)
    ymax = int(ymax + (ymax - ymin) / r)
    zmax = int(zmax + (zmax - zmin) / r)

    # Clamp to atlas volume size
    if xmin < 0:
        xmin = 0
    if xmax > ca.x_num:
        xmax = ca.x_num
    if ymin < 0:
        ymin = 0
    if ymax > ca.y_num:
        ymax = ca.y_num
    if zmin < 0:
        zmin = 0
    if zmax > ca.z_num:
        zmax = ca.z_num

    return xmin, xmax, ymin, ymax, zmin, zmax

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"

savedir = os.path.join(dst, "cfos_app")

In [ ]:
# Load atlas data

vx = 20
rdir = os.path.join(src, "CUBIC_R_atlas_ver5")

# Read atlas data
ca = cfospy.analysis.read_atlas_data(rdir, vx)
atlas_mask = ca.get_atlas_mask()
print(len(ca.ID_all))

# Get unique region IDs and their reverse mapping
uni_IDs, rev_IDs = ca.get_uni_rIDs()

# Get region summary
df_sum = ca.get_sum_temp(uni_IDs)

# Convert RGB string to normalized triplets
df_sum["rgb_triplet2"] = df_sum["rgb_triplet"].apply(lambda x: np.array(list(map(int, x.strip("[]").split(", ")))) / 255)
print(df_sum)

In [ ]:
# Read rhythmicity data

cos_dir = os.path.join(src, "cos_results")
res = "cos.cell_count_1st2nd_ai_fpr0.5.csv"

# Read cosinor test results
ct_path = os.path.join(cos_dir, res)
CT_df = pd.read_csv(ct_path)

CT_df

In [ ]:
# Voxel-based parameter setup and phase–FDR data loading

unit = "vb"
op_b = "border"   # or "noborder"
type_v = "count"

# Select calculation folder by type
if type_v == "count":
    calc_dir = f"whole_{unit}_a"
elif type_v == "count_ratio":
    calc_dir = f"whole_{unit}_cir"

op1 = "fdr"
op2 = "nuc_atlas"

vb_r = 8
mo = 1
r = 100

n = 3
b = 0.01
a = (1 - b) / (-1) ** n

dir = os.path.join(calc_dir, f"{vx}um", "whole", f"vb{vb_r}_mo{mo}")

cos_v_df = pd.read_csv(os.path.join(savedir, dir, "cos_1st2nd.csv"))
print(cos_v_df)

# Phase to HSV hue
ph_li = (cos_v_df["LAG"] / 24)
ph_li = 1 - ph_li
ph_li = [(h + 1/3 - 1) if (h + 1/3) > 1 else (h + 1/3) for h in ph_li]
color_li = [hsv_to_rgb([1, 1, h]) for h in ph_li]

# FDR to alpha (normalize then transform)
fdr_vs = np.array(cos_v_df["BH.Q"])
alpha_li = -np.log10(fdr_vs)
a_max = np.max(alpha_li)
alpha_li = alpha_li / a_max
alpha_li = (-a * (alpha_li - 1) ** n + 1)

# Brain voxel indices
brain_ind = np.where(np.ravel(np.swapaxes(atlas_mask, 0, 2)) == 1)[0]

In [ ]:
# Assign colors and merge voxel coordinates

# Non-significant voxels
non_sig_ind = cos_v_df[cos_v_df["BH.Q"] > 0.1].index

color_li = [(0.5, 0.5, 0.5) if i in non_sig_ind else color for i, color in enumerate(color_li)]
cos_v_df["color"] = color_li
cos_v_df["alpha"] = alpha_li

# Load voxel coordinates for the brain volume
vx_cords = np.load(os.path.join(rdir, f"{vx}um", "voxel_cords_brain.npy"))
vx_cords.shape
vx_cords = pd.DataFrame(vx_cords, columns=["X", "Y", "Z"])

cos_v_df = pd.concat([cos_v_df, vx_cords], axis=1)

del color_li, non_sig_ind, ph_li, fdr_vs, a_max, alpha_li, vx_cords
gc.collect()

In [ ]:
cos_v_df

In [ ]:
# Generate rotation movie frames for voxel-level phase maps

fig_dir = os.path.join(savedir, "figures_movie")
region_IDs = uni_IDs[5:]

num_frames = 6

vx = 20
vx2 = 20
elev = 30

brain_ind = np.where(np.ravel(np.swapaxes(atlas_mask, 0, 2)) == 1)[0]
brain_ind = ca.voxel_ID_order_all[brain_ind]

for i, rID in enumerate(region_IDs):
    region = ca.df_allen[ca.df_allen["ID"] == rID]["acronym"].iloc[0]
    print(rID, region)

    if "/" in region:
        region = region.replace("/", "_")

    name = f"{region}_vx"
    region_dir = os.path.join(fig_dir, name)
    os.makedirs(region_dir, exist_ok=True)

    # Collect voxel indices for the region (including children if applicable)
    if ca.smallID_q(rID):
        index = np.where(brain_ind == rID)[0].tolist()
    else:
        child_IDs, _, middle_IDs, _ = ca.get_child_IDs2(rID)
        index = np.where(brain_ind == rID)[0].tolist()
        for mID in child_IDs + middle_IDs:
            index += np.where(brain_ind == mID)[0].tolist()

    cos_v_df2 = cos_v_df.iloc[index]
    brain_IDs = [315, rID]
    xmin, xmax, ymin, ymax, zmin, zmax = mask_range(brain_IDs, 100)

    x_li = cos_v_df2["X"].tolist()
    y_li = cos_v_df2["Y"].tolist()
    z_li = cos_v_df2["Z"].tolist()
    alphas = cos_v_df2["alpha"].tolist()
    colors = cos_v_df2["color"].tolist()

    if os.path.exists(os.path.join(region_dir, f"phase_rot_{elev}_4.png")):
        continue

    c = 0
    for j, azim in enumerate(np.linspace(0, 360, num_frames)[0:5]):
        if os.path.exists(os.path.join(fig_dir, name, f"phase_rot_{elev}_{c}.png")):
            c += 1
            continue

        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection="3d")

        ax.view_init(elev, azim)
        ax.scatter(x_li, y_li, z_li, color=colors, alpha=alphas)

        ax.set_xlim(xmin * vx / vx2, xmax * vx / vx2)
        ax.set_ylim(ymin * vx / vx2, ymax * vx / vx2)
        ax.set_zlim(zmin * vx / vx2, zmax * vx / vx2)

        ax.set_zlim(ax.get_zlim()[::-1])
        ax.axis("off")

        png_path = os.path.join(fig_dir, name, f"phase_rot_{elev}_{c}.png")
        svg_path = os.path.join(fig_dir, name, f"phase_rot_{elev}_{c}.SVG")

        fig.savefig(png_path)
        fig.savefig(svg_path)
        c += 1
        plt.show()

In [ ]:
# Generate rotation frames for whole-brain phase visualization

fig_dir = os.path.join(savedir, "figures_movie_phase")

num_frames = 6
elev = 30

region_IDs = [385]
brain_IDs = [315, 698, 1089, 703, 477, 803, 549, 1097, 313, 771, 354, 512]

xmin, xmax, ymin, ymax, zmin, zmax = mask_range(brain_IDs, 100)
y_scale = 1.5

brain_masks = []
face_colors = []

for rID in brain_IDs:
    ind = ca.get_vx_ind(rID)
    brain_mask = np.zeros(ca.voxel_nums, dtype="uint16")
    brain_mask[ind] = 1
    brain_mask = np.swapaxes(brain_mask.reshape(ca.x_num, ca.y_num, ca.z_num), 0, 2)
    brain_mask = np.repeat(brain_mask, y_scale, axis=1)
    brain_masks.append(brain_mask)
    face_colors.append((0.78, 0.78, 0.78) if rID == 315 else (0.93, 0.93, 0.93))

for m, region_ID in enumerate(region_IDs):
    region = ca.df_allen[ca.df_allen["ID"] == region_ID]["acronym"].iloc[0]
    print(region, region_ID)
    if "/" in region:
        region = region.replace("/", "_")

    name = f"{region}_whole_vx"
    os.makedirs(os.path.join(fig_dir, name), exist_ok=True)

    ind = ca.get_vx_ind(region_ID)
    region_mask = np.zeros(ca.voxel_nums, dtype="uint16")
    region_mask[ind] = 1
    region_mask = np.swapaxes(region_mask.reshape(ca.x_num, ca.y_num, ca.z_num), 0, 2)

    ph = CT_df[CT_df["id"] == region_ID]["LAG"].tolist()
    ph = 1 - (ph[0] / 24)
    ph = ph + 1/3 - 1 if (ph + 1/3) > 1 else ph + 1/3
    face_color_r = hsv_to_rgb([ph, 1, 1])
    print(face_color_r)

    region_mask = np.repeat(region_mask, y_scale, axis=1)
    if np.sum(region_mask) == 0:
        continue
        print(lr, "0 mask continue")

    c = 0
    for j, azim in enumerate(np.linspace(0, 360, num_frames)):
        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection="3d")

        for brain_mask, face_color in zip(brain_masks, face_colors):
            verts, faces, _, _ = measure.marching_cubes(brain_mask, level=0.5)
            verts_swapped = verts[:, [2, 1, 0]]
            mesh = Poly3DCollection(verts_swapped[faces], alpha=0.025)
            mesh.set_facecolor(face_color)
            ax.add_collection3d(mesh)

        verts, faces, _, _ = measure.marching_cubes(region_mask, level=0.5)
        verts_swapped = verts[:, [2, 1, 0]]
        mesh = Poly3DCollection(verts_swapped[faces], alpha=1.0)
        mesh.set_facecolor(face_color_r)
        ax.add_collection3d(mesh)

        ax.view_init(elev=elev, azim=azim)
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_zlim(zmin, zmax)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        ax.set_zlim(ax.get_zlim()[::-1])
        ax.axis("off")

        png_path = os.path.join(fig_dir, name, f"phase_small_rot_{elev}_{num_frames}_{c}.png")
        fig.savefig(png_path)
        c += 1
        plt.show()
        plt.close()